# Group60 NB3 — mBERT Baseline
**Kaggle. GPU required. ~60 minutes.**

Settings → Accelerator → GPU T4 x2 → Internet On → Run All

**Outputs:** `Group60_mBERT_{lang}_*/`, `Group60_Results_mBERT_*.json`

### Step 1 — Install

In [ ]:
import subprocess
subprocess.run(['pip', 'install', '-q', 'langdetect', 'lime', 'accelerate'], check=True)
print("Packages ready.")

### Step 2 — Setup

In [ ]:
import os, random, warnings, json, shutil
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import joblib

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score, classification_report
from sklearn.pipeline import Pipeline

import torch
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer,
    EarlyStoppingCallback, DataCollatorWithPadding,
    set_seed as hf_set_seed
)
from datasets import Dataset as HFDataset
from langdetect import detect, LangDetectException

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
hf_set_seed(SEED)

LANGUAGES     = {"hau":"Hausa","yor":"Yoruba","ibo":"Igbo","pcm":"Nigerian Pidgin","swa":"Swahili"}
LABEL_MAP     = {"positive":2,"neutral":1,"negative":0}
LABEL_NAMES   = ["negative","neutral","positive"]
CMI_THRESHOLD = 0.20
MODEL_NAME    = "Davlan/afro-xlmr-base"
MBERT_NAME    = "google-bert/bert-base-multilingual-cased"
RUN_ID        = datetime.now().strftime("%Y%m%d_%H%M")

# Kaggle: /kaggle/working is the persistent output directory
# Everything saved here is downloadable from the Output panel on the right
OUTPUT_DIR = Path("/kaggle/working/group60_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE    = "cuda" if torch.cuda.is_available() else "cpu"
MAX_LEN   = 64
BATCH_SIZE= 32
GRAD_ACCUM= 1
LR        = 2e-5
N_EPOCHS  = 5
PATIENCE  = 2
RESULTS   = {code:{} for code in LANGUAGES}

def save_checkpoint(label="ckpt"):
    """Print what has been saved so far - Kaggle auto-saves /kaggle/working."""
    files = list(OUTPUT_DIR.glob("*"))
    print(f"  [{label}] {len(files)} files in output dir. Kaggle auto-saves these.")
    for f in sorted(files)[-5:]:
        print(f"    {f.name}")

def evaluate_on_subsets(predict_fn, code, model_name):
    RESULTS[code][model_name] = {}
    for sname, df in [("full",DATA[code]["test"]),
                       ("mono",DATA[code]["test_mono"]),
                       ("mixed",DATA[code]["test_mixed"])]:
        if len(df)==0: RESULTS[code][model_name][sname]=None; continue
        yt = df["label_int"].values
        yp = predict_fn(df["tweet"].tolist())
        RESULTS[code][model_name][sname] = {
            "weighted_f1": round(f1_score(yt,yp,average="weighted",zero_division=0)*100,2),
            "macro_f1":    round(f1_score(yt,yp,average="macro",   zero_division=0)*100,2),
            "accuracy":    round(accuracy_score(yt,yp)*100,2),
            "n_samples":   len(df),
            "report":      classification_report(yt,yp,target_names=LABEL_NAMES,
                                                  zero_division=0,output_dict=True),
        }
        r=RESULTS[code][model_name][sname]
        print(f"    [{sname:5s}] n={len(df):,}  wF1={r['weighted_f1']}  mF1={r['macro_f1']}")

def compute_metrics(ep):
    logits,labels = ep
    if isinstance(logits,tuple): logits=logits[0]
    preds = np.argmax(logits,axis=-1)
    return {"weighted_f1":f1_score(labels,preds,average="weighted",zero_division=0),
            "macro_f1":   f1_score(labels,preds,average="macro",   zero_division=0),
            "accuracy":   accuracy_score(labels,preds)}

def df_to_hf(df, tokenizer):
    d = HFDataset.from_pandas(df[["tweet","label_int"]].rename(columns={"label_int":"labels"}))
    def tok_fn(ex, t=tokenizer):
        return t(ex["tweet"], truncation=True, max_length=MAX_LEN, padding=False)
    d = d.map(tok_fn, batched=True, remove_columns=["tweet"])
    d.set_format("torch"); return d

def get_training_args(output_dir):
    return TrainingArguments(
        output_dir=str(output_dir),
        num_train_epochs=N_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        learning_rate=LR, weight_decay=0.01, warmup_steps=200,
        gradient_accumulation_steps=GRAD_ACCUM, lr_scheduler_type="linear",
        eval_strategy="epoch", save_strategy="epoch",
        load_best_model_at_end=True, metric_for_best_model="weighted_f1",
        greater_is_better=True, save_total_limit=1, seed=SEED,
        logging_steps=50, report_to="none",
        fp16=torch.cuda.is_available(), dataloader_num_workers=2,
    )

def run_inference(model, tokenizer, texts, batch=64):
    preds=[]; model.eval()
    for i in range(0, len(texts), batch):
        enc = tokenizer(texts[i:i+batch], truncation=True, max_length=MAX_LEN,
                        padding=True, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            preds.extend(model(**enc).logits.argmax(-1).cpu().tolist())
    return preds

def save_results_json(tag=""):
    def clean(o):
        if isinstance(o,dict): return {k:clean(v) for k,v in o.items()}
        if isinstance(o,(int,float,str,bool,type(None))): return o
        if isinstance(o,np.integer): return int(o)
        if isinstance(o,np.floating): return float(o)
        if isinstance(o,np.ndarray): return o.tolist()
        return str(o)
    path = OUTPUT_DIR / f"Group60_Results{tag}_{RUN_ID}.json"
    with open(path,"w") as f: json.dump(clean(RESULTS),f,indent=2)
    print(f"  Saved: {path.name}")
    return path

print(f"Device   : {DEVICE}")
print(f"Run ID   : {RUN_ID}")
print(f"Output   : {OUTPUT_DIR}")
print("Setup complete.")


### Step 3 — Load data + CMI

In [ ]:
GITHUB_BASE = "https://raw.githubusercontent.com/afrisenti-semeval/afrisent-semeval-2023/main/data"
SPLIT_FILES = {"train":"train.tsv","validation":"dev.tsv","test":"test.tsv"}
DATA = {}

def load_tsv(code, split):
    return pd.read_csv(f"{GITHUB_BASE}/{code}/{SPLIT_FILES[split]}",
                       sep="\t", header=0, on_bad_lines="skip")

def normalise(df):
    df=df.copy(); col_map={}
    for c in df.columns:
        if c.lower() in ("text","tweet"):        col_map[c]="tweet"
        elif c.lower() in ("label","sentiment"): col_map[c]="label"
    df=df.rename(columns=col_map)
    df["tweet"]=df["tweet"].astype(str).str.strip()
    df["label"]=df["label"].astype(str).str.strip().str.lower()
    df=df[df["label"].isin(LABEL_MAP)].copy()
    df["label_int"]=df["label"].map(LABEL_MAP)
    return df.reset_index(drop=True)

for code,name in LANGUAGES.items():
    print(f"Loading {name}...", end=" ", flush=True)
    DATA[code]={s:normalise(load_tsv(code,s)) for s in ["train","validation","test"]}
    tr,va,te=DATA[code]["train"],DATA[code]["validation"],DATA[code]["test"]
    print(f"train={len(tr):,}  val={len(va):,}  test={len(te):,}")
print("Data loaded.")

def is_valid_token(tok):
    tok=tok.strip(".,!?;:\"'()[]")
    return tok and len(tok)>=3 and not tok.startswith(("@","http")) and not tok.isdigit()

def _detect_en(tok):
    try: return detect(tok)=="en"
    except LangDetectException: return False

def english_token_ratio(tweet):
    valid=[t for t in tweet.split() if is_valid_token(t)]
    if not valid: return 0.0
    return sum(1 for t in valid if _detect_en(t)) / len(valid)

for code,name in LANGUAGES.items():
    print(f"  CMI {name}...", end=" ", flush=True)
    df=DATA[code]["test"].copy()
    df["en_ratio"]=df["tweet"].apply(english_token_ratio)
    df["code_mixed"]=df["en_ratio"]>CMI_THRESHOLD
    DATA[code]["test"]=df
    DATA[code]["test_mono"]=df[~df["code_mixed"]].reset_index(drop=True)
    DATA[code]["test_mixed"]=df[df["code_mixed"]].reset_index(drop=True)
    n,nm=len(df),df["code_mixed"].sum()
    print(f"mono={n-nm}  mixed={nm} ({100*nm/n:.1f}%)")
print("CMI done.")


### Step 4 — mBERT Training

In [ ]:
print("=== mBERT Baseline ===")
mbert_tok = AutoTokenizer.from_pretrained(MBERT_NAME)

SKIP_DONE = []  # e.g. ["hau"] to skip completed languages

for code, name in LANGUAGES.items():
    if code in SKIP_DONE:
        print(f"Skipping {name}")
        continue

    print(f"\nmBERT -> {name} ({code})")
    out_dir = OUTPUT_DIR / f"Group60_mBERT_{code}_{RUN_ID}"
    out_dir.mkdir(exist_ok=True)

    train_hf = df_to_hf(DATA[code]["train"],      mbert_tok)
    val_hf   = df_to_hf(DATA[code]["validation"], mbert_tok)

    model = AutoModelForSequenceClassification.from_pretrained(
        MBERT_NAME, num_labels=3,
        id2label={0:"negative",1:"neutral",2:"positive"},
        label2id={"negative":0,"neutral":1,"positive":2},
        ignore_mismatched_sizes=True,
    ).to(DEVICE)

    trainer = Trainer(
        model=model, args=get_training_args(out_dir),
        train_dataset=train_hf, eval_dataset=val_hf,
        processing_class=mbert_tok,
        data_collator=DataCollatorWithPadding(mbert_tok),
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)],
    )
    trainer.train()

    _m, _t = model, mbert_tok
    evaluate_on_subsets(lambda texts, m=_m, t=_t: run_inference(m,t,texts), code, "mbert")

    del model, trainer, _m, _t
    torch.cuda.empty_cache()

    save_results_json("_mBERT")
    save_checkpoint(f"mBERT {name} done")

print("\nmBERT done. Download from Output panel.")
